# Import library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Load Dataset


In [ ]:
df = pd.read_csv('train.csv')
print('Dataset loaded successfully.')
print(f'Shape: {df.shape}')

# Data Understanding

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())

In [ ]:
print('Data Types:')
print(df.dtypes)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    print(f'{col}: {df[col].unique()}')

In [ ]:
print(df['satisfaction'].value_counts())
print(df['satisfaction'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

# Data Cleaning and Preparation

In [ ]:
# Step 1: Drop irrelevant index column
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)
    print('Dropped Unnamed: 0 column.')
else:
    print('No Unnamed column found.')

In [ ]:
# Step 2: Standardize column names to snake_case
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(' ', '_')
              .str.replace('/', '_'))
print('Column names standardized.')
print(df.columns.tolist())


In [ ]:
# Step 3: Remove duplicate rows
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Removed {before - len(df)} duplicate rows. Remaining: {len(df)}')

In [ ]:

# Step 4: Fill missing values with median

median_delay = df['arrival_delay_in_minutes'].median()
df['arrival_delay_in_minutes'].fillna(median_delay, inplace=True)
print("Missing values filled with median:", median_delay)
print("Remaining nulls:", df['arrival_delay_in_minutes'].isnull().sum())

In [ ]:
# Step 5: Binary encoding of satisfaction
# 1 = satisfied, 0 = neutral or dissatisfied
df['satisfaction_binary'] = (df['satisfaction'] == 'satisfied').astype(int)

print("Encoded: 1 = satisfied, 0 = neutral or dissatisfied")
print(df[['satisfaction', 'satisfaction_binary']].head())

In [ ]:
print('Final shape:', df.shape)
print('Total missing values:', df.isnull().sum().sum())

# Feature Engineering

In [ ]:
# Feature 1: Age Group (binning age into categories)
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 30, 45, 60, 100],
    labels=['<18', '18-30', '31-45', '46-60', '60+'],
    right=True,
    include_lowest=True
)

print("Feature 1 - age_group distribution:")
print(df['age_group'].value_counts().to_string())

In [ ]:
# Feature 2: Total Delay
# Combining departure and arrival delay into a single metric captures the overall delay burden.
df['total_delay'] = df['departure_delay_in_minutes'] + df['arrival_delay_in_minutes']
print('Feature 2 - total_delay:')
print(df['total_delay'].describe().to_string())

In [ ]:
# Feature 3: Average service rating
service_cols = [
    'inflight_wifi_service', 'departure_arrival_time_convenient',
    'ease_of_online_booking', 'gate_location', 'food_and_drink',
    'online_boarding', 'seat_comfort', 'inflight_entertainment',
    'on-board_service', 'leg_room_service', 'baggage_handling',
    'checkin_service', 'inflight_service', 'cleanliness'
]

df['avg_service_rating'] = df[service_cols].mean(axis=1).round(2)

print("Average service rating created")
print(df['avg_service_rating'].describe())

In [ ]:
# Feature 4: Delay Category
# Labelled severity bands make it easier to compare satisfaction across delay groups.
df['delay_category'] = pd.cut(
    df['total_delay'],
    bins=[-1, 0, 30, 120, df['total_delay'].max()],
    labels=['No Delay', 'Minor (1-30 min)', 'Moderate (31-120 min)', 'Severe (>120 min)']
)
print('Delay_category:')
print(df['delay_category'].value_counts().to_string())

# Data Analysis

In [ ]:
# Analysis 1 — Satisfaction Rate by Class (Subgroup Comparison 1)
sat_by_class = df.groupby('class')['satisfaction_binary'].mean().mul(100).round(2)
print('Satisfaction Rate by Class (%):')
print(sat_by_class.to_string())

In [ ]:
# Analysis 2 — Satisfaction Rate by Travel Type (Subgroup Comparison 2)
sat_by_travel = df.groupby('type_of_travel')['satisfaction_binary'].mean().mul(100).round(2)
print('Satisfaction Rate by Travel Type (%):')
print(sat_by_travel.to_string())

In [ ]:
# Analysis 3 — Class x Travel Type Cross-tab
cross = df.groupby(['class', 'type_of_travel'])['satisfaction_binary'].mean().mul(100).round(2).unstack()
print('Satisfaction Rate — Class x Travel Type (%):')
print(cross.to_string())

In [ ]:
# Analysis 4: Correlation of service ratings with satisfaction
corr = df[service_cols + ['satisfaction_binary']].corr()['satisfaction_binary']

corr = corr.drop('satisfaction_binary').sort_values(ascending=False)

print("Service rating vs satisfaction correlation:")
print(corr.to_string())

In [ ]:
# Analysis 5 — Average Service Rating by Satisfaction Group
print(df.groupby('satisfaction')['avg_service_rating'].mean().round(2).to_string())

In [ ]:
# Analysis 6 — Mean Total Delay by Satisfaction Group
print(df.groupby('satisfaction')['total_delay'].mean().round(2).to_string())

In [ ]:
# Analysis 7: Outliers in total_delay (IQR method)
Q1, Q3 = df['total_delay'].quantile([0.25, 0.75])
upper_fence = Q3 + 1.5 * (Q3 - Q1)

outliers = df[df['total_delay'] > upper_fence]

print(f"IQR Upper fence: {upper_fence:.1f} min")
print(f"Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
print(outliers['satisfaction'].value_counts().to_string())

In [ ]:
# Analysis 8 — Satisfaction Rate by Age Group
print(df.groupby('age_group', observed=True)['satisfaction_binary'].mean().mul(100).round(2).to_string())

# Using NumPy 

In [ ]:
# Z-Score Normalization of avg_service_rating
# Z-scores show how far each passenger's service experience deviates from the mean.
ratings_array = df['avg_service_rating'].values
z_scores = (ratings_array - np.mean(ratings_array)) / np.std(ratings_array)
df['avg_rating_zscore'] = z_scores.round(3)

print(f'Z-score mean : {np.mean(z_scores):.4f}')
print(f'Z-score std  : {np.std(z_scores):.4f}')
print(f'Z-score min  : {np.min(z_scores):.4f}')
print(f'Z-score max  : {np.max(z_scores):.4f}')

In [ ]:
# Satisfaction rate for passengers with z-score > 1.5 (high service experience)
high_rated = df[df['avg_rating_zscore'] > 1.5]
print(f'Passengers with z-score > 1.5: {len(high_rated)}')
print(high_rated['satisfaction'].value_counts().to_string())    

In [ ]:
# Flight distance percentiles
percentiles = np.percentile(df['flight_distance'], [25, 50, 75, 90, 99])

labels = ['25th', '50th', '75th', '90th', '99th']

print("Flight Distance Percentiles:")
for label, value in zip(labels, percentiles):
    print(f"{label}: {value:.0f} km")

In [ ]:
# Satisfaction rate: top vs bottom quartile of service rating
q25, q75 = np.percentile(df['avg_service_rating'], [25, 75])

low_group  = df[df['avg_service_rating'] <= q25]
high_group = df[df['avg_service_rating'] >= q75]

low_rate  = low_group['satisfaction_binary'].mean() * 100
high_rate = high_group['satisfaction_binary'].mean() * 100

print(f"Bottom 25% — satisfaction: {low_rate:.1f}%")
print(f"Top 25%    — satisfaction: {high_rate:.1f}%")

# Data Visualizations

In [ ]:
# Figure 1 — Satisfaction Rate by Class and Travel Type
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 1 — Satisfaction Rate by Class and Travel Type', fontsize=14, fontweight='bold')

# Left — By Class
class_data = df.groupby('class')['satisfaction_binary'].mean().mul(100).round(2).sort_values(ascending=False)
axes[0].bar(class_data.index, class_data.values, color=['#0047AB', '#FF8C00', '#2ecc71'], edgecolor='white')
axes[0].set_title('By Passenger Class')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Satisfaction Rate (%)')
axes[0].set_ylim(0, 110)
for i, (cls, val) in enumerate(class_data.items()):
    axes[0].text(i, val + 1.5, f'{val:.1f}%', ha='center', fontweight='bold')

# Right — By Travel Type
travel_data = df.groupby('type_of_travel')['satisfaction_binary'].mean().mul(100).round(2).sort_values(ascending=False)
axes[1].bar(travel_data.index, travel_data.values, color=['#0047AB', '#FF8C00'], edgecolor='white')
axes[1].set_title('By Travel Type')
axes[1].set_xlabel('Travel Type')
axes[1].set_ylabel('Satisfaction Rate (%)')
axes[1].set_ylim(0, 110)
for i, (typ, val) in enumerate(travel_data.items()):
    axes[1].text(i, val + 1.5, f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('figure1_class_travel.png', bbox_inches='tight')
plt.show()
print('Business class and business travel type show the highest satisfaction rates.')

In [ ]:
# Prepare correlation data
corr_series = (
    df[service_cols + ['satisfaction_binary']]
    .corr()['satisfaction_binary']
    .drop('satisfaction_binary')
    .sort_values()
)

# Plot
fig, ax = plt.subplots(figsize=(11, 6))

bar_colors = ['#0047AB' if v >= 0 else '#FF8C00' for v in corr_series.values]

ax.barh(corr_series.index, corr_series.values, color=bar_colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)

# Labels on bars
for i, val in enumerate(corr_series.values):
    xpos = val + 0.005 if val >= 0 else val - 0.005
    ax.text(xpos, i, f'{val:.3f}',
            va='center',
            ha='left' if val >= 0 else 'right',
            fontsize=9)

# Formatting
ax.set_xlim(ax.get_xlim()[0] - 0.02, ax.get_xlim()[1] + 0.02)
ax.set_title('Figure 2 — Service Rating Correlation with Satisfaction', fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
ax.set_ylabel('Service Category')

plt.tight_layout()
plt.savefig('figure2_correlation.png', bbox_inches='tight')
plt.show()

print("Top Service : online boarding, inflight entertainment, seat comfort")

In [ ]:
# Figure 3 — Total Delay Distribution by Satisfaction Group
fig, ax = plt.subplots(figsize=(11, 5))

sat_d   = df[df['satisfaction'] == 'satisfied']['total_delay'].clip(upper=300)
unsat_d = df[df['satisfaction'] == 'neutral or dissatisfied']['total_delay'].clip(upper=300)

bins = np.linspace(0, 300, 50)

ax.hist(sat_d,   bins=bins, alpha=1, density=True, color='#0047AB', label='Satisfied')
ax.hist(unsat_d, bins=bins, alpha=1, density=True, color='#FF8C00', label='Neutral / Dissatisfied')

ax.axvline(sat_d.mean(),   color='#0047AB', linestyle='--', linewidth=1.5, label=f'Satisfied mean ({sat_d.mean():.0f} min)')
ax.axvline(unsat_d.mean(), color='#FF8C00', linestyle='--', linewidth=1.5, label=f'Unsatisfied mean ({unsat_d.mean():.0f} min)')

ax.set_title('Figure 3 — Total Delay Distribution by Satisfaction Group (clipped at 300 min)', fontweight='bold')
ax.set_xlabel('Total Delay (minutes)')
ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.savefig('figure3_delay_dist.png', bbox_inches='tight')
plt.show()
print('Both groups are right-skewed. Unsatisfied passengers have a slightly higher average delay.')

In [ ]:
# Figure 4 — Satisfaction Rate by Age Group
fig, ax = plt.subplots(figsize=(9, 5))

sat_age = df.groupby('age_group', observed=True)['satisfaction_binary'].mean().mul(100)
colors  = ["#052E67", "#C16A00", "#00a043", "#af1403", "#7900a8"]

ax.bar(sat_age.index.astype(str), sat_age.values, color=colors, edgecolor='white')

for i, val in enumerate(sat_age.values):
    ax.text(i, val + 1.5, f'{val:.1f}%', ha='center', fontweight='bold')

ax.set_title('Figure 4 — Satisfaction Rate by Age Group', fontweight='bold')
ax.set_xlabel('Age Group')
ax.set_ylabel('Satisfaction Rate (%)')
ax.set_ylim(0, 110)

plt.tight_layout()
plt.savefig('figure4_age.png', bbox_inches='tight')
plt.show()
print('Middle-aged passengers (31–60) tend to be more satisfied than younger or older groups.')

In [ ]:
# Figure 5 — Satisfaction Rate by Age Group
fig, ax = plt.subplots(figsize=(10, 5))

delay_order = ['No Delay', 'Minor (1-30 min)', 'Moderate (31-120 min)', 'Severe (>120 min)']

avg_rating_delay = (
    df.groupby('delay_category', observed=True)['avg_service_rating']
    .mean()
    .reindex(delay_order)
)

ax.plot(
    avg_rating_delay.index.astype(str),
    avg_rating_delay.values,
    marker="o",
    linewidth=2
)

ax.set_xlabel("Delay Category")
ax.set_ylabel("Avg Service Rating (1-5)")
ax.set_title('Figure 5 — Average Service Rating Trend Across Delay Categories', fontweight='bold')
ax.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig('figure5_service_rating_delay.png', bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6 — Age vs Average Service Rating by Satisfaction
fig, ax = plt.subplots(figsize=(12, 6))

ax.scatter(
    df["age"],
    df["avg_service_rating"],
    c=df["satisfaction_binary"].map({1: "#0047AB", 0: "#FF8C00"}),
    s=30,
    alpha=1,
    linewidth=0
)

m, b = np.polyfit(df["age"], df["avg_service_rating"], 1)
x_vals = np.linspace(df["age"].min(), df["age"].max(), 300)
ax.plot(x_vals, m * x_vals + b, color="maroon", linewidth=2.5)

ax.set_title("Figure 6 — Age vs Average Service Rating by Satisfaction", fontweight="bold")
ax.set_xlabel("Age")
ax.set_ylabel("Average Service Rating (0–5)")
ax.grid(True, alpha=0.25)
ax.legend(handles=[
    mpatches.Patch(color="#0047AB", label="Satisfied"),
    mpatches.Patch(color="#FF8C00", label="Neutral / Dissatisfied")
], title="Satisfaction")

plt.tight_layout()
plt.savefig("figure6_scatter_age_rating.png", bbox_inches="tight")
plt.show()
print("The trend line shows average service rating slightly increases with age.")

In [ ]:
# Figure 7 — Flight Distance vs Total Delay by Satisfaction
fig, ax = plt.subplots(figsize=(15, 10))

for label, color in [('satisfied', '#0047AB'), ('neutral or dissatisfied', '#FF8C00')]:
    subset = df[df['satisfaction'] == label]
    ax.scatter(
        subset['flight_distance'],
        subset['total_delay'].clip(upper=500),
        alpha=0.90,
        s=8,
        color=color,
        label=label.title()
    )

m, b = np.polyfit(df['flight_distance'], df['total_delay'].clip(upper=500), 1)
x_vals = np.linspace(df['flight_distance'].min(), df['flight_distance'].max(), 300)
ax.plot(x_vals, m * x_vals + b, color='maroon', linewidth=2.5, label='Trend Line')

ax.set_title('Figure 7 — Flight Distance vs Total Delay by Satisfaction', fontweight='bold')
ax.set_xlabel('Flight Distance (km)')
ax.set_ylabel('Total Delay (minutes, clipped at 500)')
ax.grid(True, alpha=0.25)
ax.legend(title='Satisfaction')

plt.tight_layout()
plt.savefig('figure7_scatter_distance_delay.png', bbox_inches='tight')
plt.show()
print('Longer flights do not consistently cause more delays. Delay is spread across all distances.')

In [ ]:
# Figure 8 — Satisfaction & Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left — Overall Satisfaction
sat_counts = df['satisfaction'].value_counts()
axes[0].pie(
    sat_counts.values,
    labels=sat_counts.index.str.title(),
    colors=['#0047AB', '#FF8C00'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=11)
)
axes[0].set_title('Overall Satisfaction Distribution', fontweight='bold')

# Right — Passenger Class
class_counts = df['class'].value_counts()
axes[1].pie(
    class_counts.values,
    labels=class_counts.index,
    colors=['#0047AB', '#FF8C00', '#2ecc71'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=11)
)
axes[1].set_title('Passenger Class Distribution', fontweight='bold')

fig.suptitle('Figure 8 — Satisfaction & Class Distribution', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('figure8_pie_charts.png', bbox_inches='tight')
plt.show()
print('57% of passengers are satisfied. Business class makes up the largest share of passengers.')

In [ ]:
# Check missing values in entire dataset
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

missing_only = missing_df[missing_df['Missing Count'] > 0]

if missing_only.empty:
    print('No missing values found in any column.')
else:
    print(f'Columns with missing values:')
    print(missing_only)